# Notebook 42 — Prelim 1 blind team-candidate packet

Runs the current SOTUYEN1 package with R5 query views, conservative A0/S1 visual head, external ASR/E5/OCR/object evidence, optional bounded XCLIP TRAKE audit, contact sheets, and A0/S1 embeddings. It never opens GT, runs Whisper, tunes from leaderboard feedback, uploads a submission, or requires the nonexistent Completion-v1.1 evidence bundle.


In [ ]:
import os
from pathlib import Path
REPO_URL=os.environ.get('AIC_REPO_URL','https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF=os.environ.get('AIC_REPO_REF','TRIAGEEG')
REPO_DIR=Path(os.environ.get('AIC_REPO_DIR','/kaggle/working/AIC2026_TeamPTK_SGU'))
QUERY_INPUT=Path(os.environ.get('AIC_PRELIM1_QUERY_ROOT','/kaggle/input/datasets/irthn1311/sotuyen1-bo-de-thi'))
RAW_INPUT=Path(os.environ.get('AIC_DATA_ROOT','/kaggle/input/datasets/nadkli/dataset-aic'))
STAGE1_INPUT=Path(os.environ.get('AIC_STAGE1_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'))
STAGE1B_INPUT=Path(os.environ.get('AIC_STAGE1B_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'))
STAGE1E_INPUT=Path(os.environ.get('AIC_STAGE1E_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze'))
CLIP_INPUT=Path(os.environ.get('AIC_CLIP_ROOT','/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32'))
OPUS_INPUT=Path(os.environ.get('AIC_OPUS_ROOT','/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en'))
SIGLIP_INPUT=Path(os.environ.get('AIC_SIGLIP2_ROOT','/kaggle/input/datasets/irthn1311/aic2026-siglip2-base-patch16-224'))
SIGLIP_INDEX_INPUT=Path(os.environ.get('AIC_SCA1_INDEX_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-sca1-siglip2-index-v01'))
ASR_INPUT=Path(os.environ.get('AIC_ASR_EXTERNAL_V3_ROOT','/kaggle/input/datasets/irthn1311/asr-external-v3-validated-bundle'))
E5_INPUT=Path(os.environ.get('AIC_E5_ROOT','/kaggle/input/datasets/irthn1311/aic2026-multilingual-e5-small-onnx-query-encoder'))
EXTERNAL_INPUT=Path(os.environ.get('AIC_EXTERNAL_EVIDENCE_ROOT','/kaggle/input/datasets/irthn1311/external-multimodal-runtime-evidence-v3'))
XCLIP_INPUT=Path(os.environ.get('AIC_XCLIP_ROOT','/kaggle/input/datasets/irthn1311/fs1-xclip-base-patch32-asset'))
OUTPUT_ROOT=Path('/kaggle/working/prelim1_team_candidates')
OUTPUT_ZIP=Path('/kaggle/working/prelim1_team_candidates_bundle.zip')
print({'required_inputs':{'official_sotuyen1_package':str(QUERY_INPUT),'raw_dataset':str(RAW_INPUT),'stage1_exact_index':str(STAGE1_INPUT),'stage1b_contract':str(STAGE1B_INPUT),'stage1e_language':str(STAGE1E_INPUT),'openai_clip':str(CLIP_INPUT),'opus_mt':str(OPUS_INPUT),'siglip2_asset':str(SIGLIP_INPUT),'siglip2_index':str(SIGLIP_INDEX_INPUT),'external_asr_v3':str(ASR_INPUT),'e5_query_encoder':str(E5_INPUT),'external_ocr_object':str(EXTERNAL_INPUT)},'optional_input':{'xclip_asset':str(XCLIP_INPUT)},'internet_required':'GIT_CLONE_OR_REFRESH; PYTHON_DEPENDENCIES_ONLY_IF_MISSING','model_download_required':False,'whisper_run':False,'gt_opened':False,'auto_submit':False,'output_zip':str(OUTPUT_ZIP)})


In [ ]:
import subprocess,sys
if not (REPO_DIR/'.git').is_dir(): subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(['git','fetch','--no-tags','origin',REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
required=[REPO_DIR/'scripts/run_prelim1_team_candidates.py',REPO_DIR/'scripts/team_candidate_knn_consensus.py',REPO_DIR/'src/triage_eg/prelim1_team/parser.py']
missing=[str(path) for path in required if not path.is_file()]
if missing: raise RuntimeError(f'PRELIM1_TEAM_SOURCE_MISSING_FROM_RESOLVED_REF:{missing}')
sys.path.insert(0,str(REPO_DIR/'src')); sys.path.insert(0,str(REPO_DIR/'scripts'))
print({'source_ref':REPO_REF,'HEAD':HEAD,'checkout_mode':'DETACHED_FETCH_HEAD','git_status':subprocess.check_output(['git','status','--short'],cwd=REPO_DIR,text=True).strip() or 'CLEAN'})


In [ ]:
import importlib.util,re
missing=[]
for module,spec in [('onnxruntime','onnxruntime==1.20.1'),('tokenizers','tokenizers==0.21.0'),('pyarrow','pyarrow>=17,<22')]:
    if importlib.util.find_spec(module) is None: missing.append(spec)
if missing: subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',*missing],check=True)
env=os.environ.copy(); env['PYTHONPATH']=str(REPO_DIR/'src')+os.pathsep+str(REPO_DIR/'scripts')+(os.pathsep+env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
test=subprocess.run([sys.executable,'-m','pytest','tests/unit/prelim1_team','tests/unit/prelim_r5','tests/unit/sca1_siglip2_complementarity','-q'],cwd=REPO_DIR,env=env,capture_output=True,text=True)
text=test.stdout+'\n'+test.stderr
summary={'returncode':test.returncode,'passed':int(re.search(r'(\d+) passed',text).group(1)) if re.search(r'(\d+) passed',text) else None,'tail':text.splitlines()[-25:]}
print({'dependency_install':missing or 'NOT_REQUIRED','tests':summary})
if test.returncode: raise RuntimeError('PRELIM1_TEAM_REQUIRED_TESTS_FAILED')


In [ ]:
import zipfile
from run_prelim_r5_final import _find_root,_materialize_archive_root,_resolve_dataset,_resolve_mount
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root
WORK=Path('/kaggle/working/prelim1_team_inputs'); WORK.mkdir(parents=True,exist_ok=True)
query_mount=_resolve_mount(QUERY_INPUT,('SOTUYEN1-bo-de-thi','sotuyen1_bo_de_thi'))
query_zips=sorted(query_mount.rglob('SOTUYEN1-bo-de-thi.zip'))
ALLOW_REPACKED=False
if len(query_zips)==1: QUERY_ZIP=query_zips[0]
elif not query_zips:
    query_files=sorted(path for path in query_mount.rglob('query-p1-*.txt') if path.is_file())
    if len(query_files)!=25: raise RuntimeError(f'PRELIM1_QUERY_DISCOVERY_FAILED:{query_zips}:{len(query_files)}')
    QUERY_ZIP=WORK/'SOTUYEN1-bo-de-thi-repacked.zip'
    with zipfile.ZipFile(QUERY_ZIP,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in query_files: archive.write(path,path.name)
    ALLOW_REPACKED=True
else: raise RuntimeError(f'PRELIM1_QUERY_ZIP_AMBIGUOUS:{query_zips}')
DATASET_ROOT=_resolve_dataset(RAW_INPUT)
mounts={name:_resolve_mount(path,aliases) for name,path,aliases in [('stage1',STAGE1_INPUT,()),('stage1b',STAGE1B_INPUT,()),('stage1e',STAGE1E_INPUT,()),('clip',CLIP_INPUT,()),('opus',OPUS_INPUT,()),('siglip',SIGLIP_INPUT,()),('siglip_index',SIGLIP_INDEX_INPUT,()),('asr',ASR_INPUT,('asr-external-v3-validated',)),('e5',E5_INPUT,()),('external',EXTERNAL_INPUT,())]}
STAGE1_ROOT=resolve_stage1_root(mounts['stage1'],search_root=None,materialize_root=WORK/'stage1')
STAGE1B_ROOT,_=resolve_input_root(mounts['stage1b'],required=('stage1b_summary.json','encoder/selected_encoder_contract.json','encoder/runtime_adapter_manifest.json'),materialize_root=WORK/'stage1b',search_root=None,archive_keyword='stage1b')
STAGE1E_ROOT,_=resolve_input_root(mounts['stage1e'],required=('stage1e_summary.json','language_path_contract.json'),materialize_root=WORK/'stage1e',search_root=None,archive_keyword='stage1e')
CLIP_ROOT,_=resolve_input_root(mounts['clip'],required=('checkpoint/ViT-B-32.pt','manifests/asset_manifest.json'),materialize_root=WORK/'clip',search_root=None,archive_keyword='clip')
OPUS_ROOT,_=resolve_input_root(mounts['opus'],required=('model/config.json','manifests/asset_manifest.json'),materialize_root=WORK/'opus',search_root=None,archive_keyword='opus')
SIGLIP_ROOT,_=resolve_input_root(mounts['siglip'],required=('model/model.safetensors','manifests/asset_manifest.json'),materialize_root=WORK/'siglip',search_root=None,archive_keyword='siglip2')
SIGLIP_INDEX_ROOT=_materialize_archive_root(mounts['siglip_index'],'index/siglip2_vectors.f16.npy',('triage_eg_sca1_siglip2_index_v01.zip',),WORK/'siglip_index')
ASR_ROOT=_find_root(mounts['asr'],'asr_external_v3_provenance.json')
E5_ROOT=_find_root(mounts['e5'],'model.onnx')
EXTERNAL_ROOT=_find_root(mounts['external'],'ocr_records_external_v3.parquet')
xclip_mounts=[]
try: xclip_mounts=[_resolve_mount(XCLIP_INPUT)]
except RuntimeError: pass
XCLIP_ROOT=_find_root(xclip_mounts[0],'config.json') if xclip_mounts else None
resolved={'query_zip':str(QUERY_ZIP),'query_repacked':ALLOW_REPACKED,'raw':str(DATASET_ROOT),'stage1':str(STAGE1_ROOT),'stage1b':str(STAGE1B_ROOT),'stage1e':str(STAGE1E_ROOT),'clip':str(CLIP_ROOT),'opus':str(OPUS_ROOT),'siglip':str(SIGLIP_ROOT),'siglip_index':str(SIGLIP_INDEX_ROOT),'asr':str(ASR_ROOT),'e5':str(E5_ROOT),'external':str(EXTERNAL_ROOT),'xclip':str(XCLIP_ROOT) if XCLIP_ROOT else None}
print({'resolved_inputs':resolved})


In [ ]:
cmd=[sys.executable,str(REPO_DIR/'scripts/run_prelim1_team_candidates.py'),'--query-zip',str(QUERY_ZIP),'--dataset-root',str(DATASET_ROOT),'--stage1-root',str(STAGE1_ROOT),'--stage1b-root',str(STAGE1B_ROOT),'--stage1e-root',str(STAGE1E_ROOT),'--clip-root',str(CLIP_ROOT),'--opus-root',str(OPUS_ROOT),'--siglip-root',str(SIGLIP_ROOT),'--siglip-index-root',str(SIGLIP_INDEX_ROOT),'--asr-root',str(ASR_ROOT),'--e5-root',str(E5_ROOT),'--external-evidence-root',str(EXTERNAL_ROOT),'--repo-dir',str(REPO_DIR),'--output-dir',str(OUTPUT_ROOT),'--output-zip',str(OUTPUT_ZIP)]
if ALLOW_REPACKED: cmd.append('--allow-repacked-query-zip')
if XCLIP_ROOT: cmd.extend(['--xclip-root',str(XCLIP_ROOT)])
run=subprocess.run(cmd,cwd=REPO_DIR,env=env,text=True)
if run.returncode: raise RuntimeError(f'PRELIM1_TEAM_RUNNER_FAILED:{run.returncode}')


In [ ]:
import hashlib,json
if not OUTPUT_ZIP.is_file(): raise RuntimeError(f'PRELIM1_TEAM_ZIP_MISSING:{OUTPUT_ZIP}')
validation=json.loads((OUTPUT_ROOT/'packet_validation.json').read_text())
if validation['status']!='PASS': raise RuntimeError(validation)
digest=hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
manifest=json.loads((OUTPUT_ROOT/'query_manifest.json').read_text())
print({'TEAM_PACKET_READY':True,'query_package_sha256':manifest['official_package_sha256'],'resolved_archive_sha256':manifest['source_zip_sha256'],'query_content_sha256':manifest['content_sha256'],'parsed_task_counts':manifest['task_counts'],'contact_sheets_count':validation['contact_sheet_count'],'DOWNLOAD_ZIP':str(OUTPUT_ZIP),'size_bytes':OUTPUT_ZIP.stat().st_size,'sha256':digest,'gt_opened':False,'leaderboard_used':False,'submission_uploaded':False})
